In [2]:
from functools import partial
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
import rasterio.features
import rasterio.enums
import rasterio.transform
import shapely
import snail.intersection
import snail.io
from tqdm.auto import tqdm


import robyn_river_floods

In [2]:
import importlib
importlib.reload(robyn_river_floods)

<module 'robyn_river_floods' from '/home/mert2014/projects/jamaica-infrastructure/scripts/nbs_methods/river-catchment/robyn_river_floods.py'>

In [2]:
tqdm.pandas()

In [3]:
base_path = Path("../../../processed_data/nbs-river-catchment")
basin_info_dir = base_path / "upstream_catchment_info"
basins_dir = base_path / "upstream_basins_100"
damages_dir = base_path / "direct_damages"
base_path.resolve()

PosixPath('/home/mert2014/projects/jamaica-infrastructure/processed_data/nbs-river-catchment')

### Rasterise landuse with afforestable/forest proportions

In [4]:
# r.watershed drainage-direction raster (EPSG:3448)
drainage_tif = base_path / "drainage_direction_100.tif"
flood_tif = base_path / ".." / "hazards" / "Global Flood Map" / "Jamaica" / "Fluvial" / "Raw Depths" / "JM_FLRF_UD_Q500_RD_02-aligned.tif"

# Use drainage as reference raster
ref_raster = drainage_tif  # reference grid (CRS, extent, res)

ffe_path = basin_info_dir / "forest_flood_equivalent_values.tif"
aff_path = basin_info_dir / "afforestable_values.tif"

landuse_points_path = basin_info_dir / "aff_points_with_ffe.geoparquet"

In [6]:
# Read land use polygons
# assign proportion of landuse that is "forest" or "afforestable"
land_use = gpd.read_file(base_path /".."/ "nbs" / "Terrestrial Land Cover (From Forestry Department)" /"2013_landuse_Landcover.shp")
land_use["forest_flood_equivalent_values"] = (
    land_use["Classify"].map(robyn_river_floods.LANDUSE_FOREST_PROPORTION).fillna(0).astype(float)
)
land_use["afforestable_values"] = (
    land_use["Classify"].map(robyn_river_floods.LANDUSE_AFFORESTABLE_PROPORTION).fillna(0).astype(float)
)
land_use.to_file(base_path / "land_use_forest_and_afforestable.gpkg", layer="land_use", driver="GPKG")

In [7]:
# Rasterise "forest" and "afforestable" landuse
nodata = -9999.0

with rasterio.open(ref_raster) as src:
    profile = src.profile
    transform = src.transform
    shape = (src.height, src.width)
    crs = src.crs

profile.update(dtype="float32", count=1, compress="lzw", nodata=nodata)

land_use = land_use.to_crs(crs)

# Rasterize and write out forest_flood_equivalent
ffe_shapes = list(
    zip(
        land_use.geometry, land_use["forest_flood_equivalent_values"]
    )
)
ffe_arr = rasterio.features.rasterize(
    ffe_shapes,
    out_shape=shape,
    transform=transform,
    fill=nodata,
    dtype="float32",
    all_touched=False,
    merge_alg=rasterio.enums.MergeAlg.replace
)
with rasterio.open(ffe_path, "w", **profile) as dst:
    dst.write(ffe_arr, 1)

# Repeat for afforestable
aff_shapes = list(
    zip(
        land_use.geometry, land_use["afforestable_values"]
    )
)
aff_arr = rasterio.features.rasterize(
    aff_shapes,
    out_shape=shape,
    transform=transform,
    fill=nodata,
    dtype="float32",
    all_touched=False,
    merge_alg=rasterio.enums.MergeAlg.replace
)
with rasterio.open(aff_path, "w", **profile) as dst:
    dst.write(aff_arr, 1)

### Convert afforestable/forest landuse raster to representative points

In [8]:
# Read two rasters to points
with rasterio.open(aff_path) as aff, rasterio.open(ffe_path) as ffe:
    # sanity: same grid
    assert (aff.width, aff.height) == (ffe.width, ffe.height)
    assert aff.transform == ffe.transform
    assert aff.crs == ffe.crs
    px_area_m2 = abs(aff.transform.a * aff.transform.e)  # e.g., 30 * 30 = 900

    aff_arr = aff.read(1)
    ffe_arr = ffe.read(1)
    nodata = aff.nodata

    # choose which pixels to make into points
    mask = aff_arr != nodata

    rows, cols = np.where(mask)
    xs, ys = rasterio.transform.xy(aff.transform, rows, cols, offset="center")

    pts = gpd.GeoDataFrame(
        {
            # keep grid indices for joins later (match your naming)
            "dem_i": rows,   # row index
            "dem_j": cols,   # col index
            "afforestable": aff_arr[rows, cols].astype("float32"),
            "afforestable_m2": aff_arr[rows, cols].astype("float32") * px_area_m2,
            "existing_forest": ffe_arr[rows, cols].astype("float32"),
            "existing_forest_m2": ffe_arr[rows, cols].astype("float32") * px_area_m2,
        },
        geometry=gpd.points_from_xy(xs, ys),
        crs=aff.crs
    )

pts.to_parquet(landuse_points_path)

### Calculate catchment afforestable/forest proportions

In [9]:
catchments = gpd.read_parquet(base_path / "catchments.parquet")

In [17]:
def points_in_polygon(geom, all_points):
    shapely.prepare(geom)
    idx = list(all_points.sindex.query(geom, predicate="contains"))
    candidates = all_points.iloc[idx]
    return candidates[geom.contains(candidates.geometry)]

In [12]:
def catchment_forest_percentage_change(catchment, all_landuse):
    geom = catchment.geometry
    points_in_catchment = points_in_polygon(geom, all_landuse)
    # Calculate % change in forest cover
    basin_m2 = geom.area
    aff_m2 = points_in_catchment["afforestable_m2"].sum()
    ffe_m2 = points_in_catchment["existing_forest_m2"].sum()

    pct_aff   = 100 * aff_m2 / basin_m2
    return aff_m2, ffe_m2, pct_aff

upstream_basin_cover = catchments.copy()
func = partial(catchment_forest_percentage_change, all_landuse=pts)
upstream_basin_cover[["afforestable_m2", "existing_forest_m2", "afforestable_pct"]] = catchments.progress_apply(
    func,
    axis=1,
    result_type='expand'
)

  0%|          | 0/91468 [00:00<?, ?it/s]

In [13]:
upstream_basin_cover['basin_m2'] = upstream_basin_cover.geometry.area
upstream_basin_cover.head()

,geometry,dem_i,dem_j,afforestable_m2,existing_forest_m2,afforestable_pct,basin_m2
0,"POLYGON ((660739.566 703662.874, 660739.566 70...",1894,13,473850.0,4813650.0,8.003953,5920200.0
1,"POLYGON ((660739.566 703662.874, 660739.566 70...",1895,14,473850.0,4813650.0,8.006387,5918400.0
2,"POLYGON ((662059.566 707442.874, 662059.566 70...",1914,15,0.0,143100.0,0.000000,336600.0
3,"POLYGON ((660349.566 704922.874, 660349.566 70...",1857,14,0.0,2249100.0,0.000000,3663900.0
4,"POLYGON ((660349.566 704922.874, 660349.566 70...",1859,12,0.0,2249100.0,0.000000,3695400.0


In [14]:
upstream_basin_cover.to_parquet((base_path / "catchments_with_cover.parquet"))

### Calculate return period change for each catchment forest coverage 

In [12]:
upstream_basin_cover = gpd.read_parquet((base_path / "catchments_with_cover.parquet"))

In [13]:
rp_cols = ["rp5.0","rp10.0","rp20.0","rp50.0","rp100.0"]
catchment_peak_flow_reduction = robyn_river_floods.peak_flow_reduction(upstream_basin_cover.afforestable_pct) \
    [rp_cols] \
    .rename(columns={rp_col: f"pfr_{rp_col}" for rp_col in rp_cols})

catchments_with_rp_change = upstream_basin_cover.join(catchment_peak_flow_reduction)

for rp_col in rp_cols:
    rp_change = robyn_river_floods.rp_change_given_flow_reduction(reduction_percent=catchments_with_rp_change[f"pfr_{rp_col}"], interp_rp=float(rp_col.replace("rp", ""))) \
        [[rp_col]] \
        .rename(columns={rp_col: f"future_{rp_col}"})
    catchments_with_rp_change = catchments_with_rp_change.join(rp_change)

catchments_with_rp_change.iloc[0]

geometry              POLYGON ((660739.5656 703662.8742, 660739.5656...
dem_i                                                              1894
dem_j                                                                13
afforestable_m2                                                473850.0
existing_forest_m2                                            4813650.0
afforestable_pct                                               8.003953
basin_m2                                                      5920200.0
pfr_rp5.0                                                      5.504941
pfr_rp10.0                                                     5.360126
pfr_rp20.0                                                     5.070497
pfr_rp50.0                                                     4.201607
pfr_rp100.0                                                    2.753459
future_rp5.0                                                   6.742372
future_rp10.0                                                 13

In [14]:
catchments_with_rp_change.to_parquet(base_path / "catchments_with_rp_change.parquet")

### Read in split damages
- read split damages
- read flooded points snapped to river network
- for each catchment in catchments_with_rp_change
  - find the set of damaged (split) assets at any flooded point (flood_i/j) which was snapped to the outlet point (dem_i/j) of this catchment - few rows dataframe subset of split damages
  - interpolate "future" damages according to RP change due to afforestation

In [7]:
# Read all damages, filter to two (min/max) sensitivity parameter settings from ensemble_members
damage = pd.read_parquet(
    base_path / "all_damage_minmax.parquet",
    filters=[
        [('ensemble_member', '==', '7')],
        [('ensemble_member', '==', '10')],
    ],
    engine="pyarrow"
).rename(columns={
    "cell_index_2_x": "flood_i",
    "cell_index_2_y": "flood_j"
})
# Sense check a single asset / cell index (may have multiple parts, should have values for both ensemble_members)
damage.query('asset_id == "roade_83238" and flood_i == 7825').sort_values(by=['flood_i', 'flood_j', 'asset_id', 'asset_class'])

,asset_id,exposure_unit,damage_cost_unit,exposure,flood_i,flood_j,damage_uncertainty_parameter,cost_uncertainty_parameter,fluvial__rp_20__rcp_baseline__epoch_2010__conf_None,fluvial__rp_50__rcp_baseline__epoch_2010__conf_None,fluvial__rp_100__rcp_baseline__epoch_2010__conf_None,fluvial__rp_200__rcp_baseline__epoch_2010__conf_None,fluvial__rp_500__rcp_baseline__epoch_2010__conf_None,fluvial__rp_1500__rcp_baseline__epoch_2010__conf_None,asset_class,ensemble_member
1960179,roade_83238,m,J$,1.179337,7825,2160,1.0,1.0,0.0,0.0,0.0,0.0,0.0,7987.058883,roads_edges,7
1960200,roade_83238,m,J$,3.974657,7825,2160,1.0,1.0,0.0,0.0,0.0,0.0,0.0,26918.374398,roads_edges,7
1960238,roade_83238,m,J$,10.630630,7825,2160,1.0,1.0,0.0,0.0,0.0,0.0,0.0,71995.962213,roads_edges,7
1960255,roade_83238,m,J$,14.139292,7825,2160,1.0,1.0,0.0,0.0,0.0,0.0,0.0,95758.380144,roads_edges,7
2272852,roade_83238,m,J$,1.179337,7825,2160,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4564.033648,roads_edges,10
2272873,roade_83238,m,J$,3.974657,7825,2160,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15381.928227,roads_edges,10
2272911,roade_83238,m,J$,10.630630,7825,2160,0.0,0.0,0.0,0.0,0.0,0.0,0.0,41140.549836,roads_edges,10
2272928,roade_83238,m,J$,14.139292,7825,2160,0.0,0.0,0.0,0.0,0.0,0.0,0.0,54719.074368,roads_edges,10


In [8]:
# Sense check parameter and ensemble selection - should be 0/0 and 1/1 with approximately equal number of assets
damage.groupby(
    ['damage_uncertainty_parameter', 'cost_uncertainty_parameter', 'ensemble_member']
).agg(
    {'asset_id': 'count'}
)

,,,asset_id
damage_uncertainty_parameter,cost_uncertainty_parameter,ensemble_member,
0.0,0.0,10,1139715
1.0,1.0,7,1139804


### Read flooded points snapped to river network

In [9]:
# flood_i, flood_j, geometry => link to assets
flooded_points = pd.read_parquet(base_path / "non_snapped_points_with_flood_indices.parquet")
# dem_i, dem_j, geometry => link to catchment
flooded_points_snapped = pd.read_parquet(base_path / "points_to_catchment.parquet")
# dem_i, dem_j, flood_i, flood_j
flood_ij_to_dem_ij = pd.read_parquet(base_path / "flood_ij_to_dem_ij_relation.parquet")

In [10]:
# Merge damages, set index for lookup by (dem_i, dem_j) catchment index
damage_lookup = damage.merge(
    flood_ij_to_dem_ij,
    how='left',
    on=['flood_i','flood_j'],
    validate='many_to_one',
).set_index(
    ['ensemble_member', 'dem_i', 'dem_j']
)
damage_lookup.sort_index(inplace=True)
damage_lookup.loc[("7", 7684, 2223)]

asset_id exposure_unit damage_cost_unit  \
ensemble_member dem_i dem_j                                               
7               7684  2223   roade_83238             m               J$   
                      2223   roade_83238             m               J$   
                      2223   roade_83238             m               J$   
                      2223   roade_83238             m               J$   

                              exposure  flood_i  flood_j  \
ensemble_member dem_i dem_j                                
7               7684  2223    1.179337     7825     2160   
                      2223    3.974657     7825     2160   
                      2223   10.630630     7825     2160   
                      2223   14.139292     7825     2160   

                             damage_uncertainty_parameter  \
ensemble_member dem_i dem_j                                 
7               7684  2223                            1.0   
                      2223                            1.0   
                      2223                            1.0   
                      2223                            1.0   

                             cost_uncertainty_parameter  \
ensemble_member dem_i dem_j                               
7               7684  2223                          1.0   
                      2223                          1.0   
                      2223                          1.0   
                      2223                          1.0   

                             fluvial__rp_20__rcp_baseline__epoch_2010__conf_None  \
ensemble_member dem_i dem_j                                                        
7               7684  2223                                                 0.0     
                      2223                                                 0.0     
                      2223                                                 0.0     
                      2223                                                 0.0     

                             fluvial__rp_50__rcp_baseline__epoch_2010__conf_None  \
ensemble_member dem_i dem_j                                                        
7               7684  2223                                                 0.0     
                      2223                                                 0.0     
                      2223                                                 0.0     
                      2223                                                 0.0     

                             fluvial__rp_100__rcp_baseline__epoch_2010__conf_None  \
ensemble_member dem_i dem_j                                                         
7               7684  2223                                                 0.0      
                      2223                                                 0.0      
                      2223                                                 0.0      
                      2223                                                 0.0      

                             fluvial__rp_200__rcp_baseline__epoch_2010__conf_None  \
ensemble_member dem_i dem_j                                                         
7               7684  2223                                                 0.0      
                      2223                                                 0.0      
                      2223                                                 0.0      
                      2223                                                 0.0      

                             fluvial__rp_500__rcp_baseline__epoch_2010__conf_None  \
ensemble_member dem_i dem_j                                                         
7               7684  2223                                                 0.0      
                      2223                                                 0.0      
                      2223                                                 0.0      
                      2223               

### Estimate avoided damages

In [33]:
def calculate_damages_at_point(damage_subset, catchment):
    damage_cols_to_rp = {
        'fluvial__rp_20__rcp_baseline__epoch_2010__conf_None': 'rp20.0',
        'fluvial__rp_50__rcp_baseline__epoch_2010__conf_None': 'rp50.0',
        'fluvial__rp_100__rcp_baseline__epoch_2010__conf_None': 'rp100.0',
        'fluvial__rp_200__rcp_baseline__epoch_2010__conf_None': 'rp200.0',
        'fluvial__rp_500__rcp_baseline__epoch_2010__conf_None': 'rp500.0',
        'fluvial__rp_1500__rcp_baseline__epoch_2010__conf_None': 'rp1500.0',
    }
    damage_subset_rps = damage_subset[damage_cols_to_rp.keys()].rename(columns=damage_cols_to_rp)
    damage_subset_rps['rp0.0001'] = 0.0
    damage_subset_rps['rp2.0'] = 0.0
    damage_subset_rps["rp1000000000.0"] = damage_subset_rps['rp1500.0']

    interpolated_baseline_damages = robyn_river_floods.interpolate_rp_damages(
        [5.0, 10.0],
        damage_subset_rps
    )

    baseline_damages = damage_subset_rps.join(interpolated_baseline_damages)
    baseline_damages = baseline_damages[sorted(
        baseline_damages.columns,
        key=lambda c: float(c.replace("rp", ""))
    )].copy()
    # print("BASELINE", baseline_damages.iloc[11])

    future_rps = [float(colname.replace("future_rp", "")) for colname in catchment.index if "future_rp" in colname]
    future_rp_columns = [f"rp{rp}" for rp in future_rps]
    # get the *future* RP values corresponding to each
    current_to_future_rps = {
        rp: catchment[f"future_rp{float(rp)}"]
        for rp in future_rps
    }
    future_damages = baseline_damages[future_rp_columns].rename(columns={
        f"rp{current}": f"rp{future}"
        for current, future in current_to_future_rps.items()
    })
    extreme_damages = baseline_damages[["rp0.0001", "rp2.0", "rp200.0", "rp500.0", "rp1500.0", "rp1000000000.0"]].copy()
    future_damages = future_damages.join(extreme_damages)
    future_damages_columns = sorted(
        future_damages.columns,
        key=lambda c: float(c.replace("rp", ""))
    )
    future_damages = future_damages[future_damages_columns].copy()
    # at this point future damages contains columns for each of the adjusted future return periods
    # corresponding to the initial baseline set (e.g. rp6.4, rp12.0, rp24.1 corresponding to what was
    # rp5.0, rp10.0, rp20.0)

    # calculate baseline and future EAD based on all available RPs
    # print("BASELINE", baseline_damages.iloc[11])
    baseline_ead_values = robyn_river_floods.calculate_ead(baseline_damages)
    baseline_ead_colname = f"baseline__fluvial__ead"

    # interpolate to find "standard" set of RPs up to 100y
    # print("FUTURE_DMG", future_damages.iloc[11])
    interpolated_future_damages = robyn_river_floods.interpolate_rp_damages(
        future_rps,
        future_damages
    )
    # interpolation can go badly if there's no information under e.g. 100y
    # e.g. because afforestation pushed all "future" RPs to large numbers
    # so enforce that each interpolated_future_damages has the minimum of
    # current/future RP damages (up to 100y)
    for rp in future_rps:
        # print("Check", rp)
        # print(interpolated_future_damages[f"rp{rp}"].values)
        # print(baseline_damages[f"rp{rp}"].values)
        interpolated_future_damages[f"rp{rp}"] = np.min(np.array([interpolated_future_damages[f"rp{rp}"], baseline_damages[f"rp{rp}"]]), axis=0)
        # print(interpolated_future_damages[f"rp{rp}"].values)

    # print("INTERP", interpolated_future_damages.iloc[0])
    # include baseline damages as unchanged for RP > 100 (assuming NbS are unlikely to affect the extremes)
    future_rps_for_ead = interpolated_future_damages.join(extreme_damages)
    # print("FUTURE_RPS", future_rps_for_ead.iloc[0])
    future_ead_values = robyn_river_floods.calculate_ead(future_rps_for_ead)

    # now rename *output* columns (one per rp_to_calculate)
    to_rename = {
        f"rp{rp}": f"future__fluvial__rp_{int(rp)}"
        for rp in future_rps
    }
    interpolated_future_damages.rename(columns=to_rename, inplace=True)
    # pick out the interpolated future rp damages
    interpolated_future_damages = interpolated_future_damages[to_rename.values()]

    future_ead_colname = f"future__fluvial__ead"
    # add future ead as calculated above
    interpolated_future_damages[future_ead_colname] = future_ead_values

    # rename from rpXX columns to baseline...
    to_rename = {
        f"rp{rp}": f"baseline__fluvial__rp_{int(rp)}"
        for rp in future_rps
    }
    baseline_damages_renamed = baseline_damages.rename(columns=to_rename)[to_rename.values()]
    baseline_damages_renamed[baseline_ead_colname] = baseline_ead_values

    # concatenate the interpolated and baseline RP and EAD damages
    interpolated_future_damages = pd.concat([
        interpolated_future_damages,
        baseline_damages_renamed
    ], axis=1)

    # add back useful ID/sector columns from input
    interpolated_future_damages["ensemble_member"] = damage_subset["ensemble_member"]
    interpolated_future_damages["asset_id"] = damage_subset["asset_id"]
    interpolated_future_damages["asset_class"] = damage_subset["asset_class"]
    interpolated_future_damages["dem_i"] = damage_subset["dem_i"]
    interpolated_future_damages["dem_j"] = damage_subset["dem_j"]
    interpolated_future_damages["flood_i"] = damage_subset["flood_i"]
    interpolated_future_damages["flood_j"] = damage_subset["flood_j"]

    return interpolated_future_damages.copy()

# test with a particular catchment
# catchment = catchments_with_rp_change.query("dem_i == 97 and dem_j == 906").iloc[0]
# catchment = catchments_with_rp_change.query("dem_i == 1733 and dem_j == 192").iloc[0]
# catchment = catchments_with_rp_change.iloc[20000]
# catchment = catchments_with_rp_change.iloc[5000]
# dem_i, dem_j = catchment.dem_i, catchment.dem_j
# subset_damages = damage_lookup.loc[("7", dem_i, dem_j)].reset_index()

# future_damages = calculate_damages_at_point(subset_damages, catchment)

# print(f"Afforestable: {catchment.afforestable_pct:.2f}%")
# print(f"Forest: {catchment.existing_forest_m2}")
# print(f"Afforestable: {catchment.afforestable_m2}")
# print(f"Damage reduction: {100 * (future_damages.future__fluvial__ead.sum() - future_damages.baseline__fluvial__ead.sum()) / future_damages.baseline__fluvial__ead.sum():.2f}%")
# catchment, future_damages[["future__fluvial__ead", "baseline__fluvial__ead"]].sum()
# future_damages
# subset_damages

#### Run for all catchments

In [34]:
future_damage_dfs = []
for ensemble_member in ("7", "10"):
    damage_lookup_em = damage_lookup.loc[ensemble_member]
    for ci in tqdm(range(len(catchments_with_rp_change)), total=len(catchments_with_rp_change), desc=ensemble_member):
        catchment = catchments_with_rp_change.iloc[ci]
        dem_i, dem_j = catchment.dem_i, catchment.dem_j
        try:
            subset_damages = damage_lookup_em.loc[(dem_i, dem_j)].reset_index()
        except KeyError:
            print("No damages at", (dem_i, dem_j))
            continue
        subset_damages["ensemble_member"] = ensemble_member
        catchment_future_damage = calculate_damages_at_point(subset_damages, catchment)
        future_damage_dfs.append(catchment_future_damage)

future_damage = pd.concat(future_damage_dfs, axis=0)

7:   0%|          | 0/91468 [00:00<?, ?it/s]

10:   0%|          | 0/91468 [00:00<?, ?it/s]

No damages at (np.int64(4987), np.int64(869))
No damages at (np.int64(5058), np.int64(1123))
No damages at (np.int64(5324), np.int64(1462))
No damages at (np.int64(5872), np.int64(1889))
No damages at (np.int64(6996), np.int64(2281))


In [35]:
future_damage.to_parquet(base_path / "damage__future.parquet")

In [36]:
damages_by_flooded_point = (
    future_damage
    .groupby(["ensemble_member", "flood_i","flood_j"])
    .agg({
        "dem_i": "first",
        "dem_j":"first",
        "baseline__fluvial__ead": "sum",
        "future__fluvial__ead": "sum",
    })
)
damages_by_flooded_point["avoided__fluvial__ead"] = damages_by_flooded_point.baseline__fluvial__ead - damages_by_flooded_point.future__fluvial__ead

In [37]:
damages_by_flooded_point.to_parquet(base_path / "damages_by_flooded_point__future.parquet")

In [38]:
def splits_to_raster(
        splits: pd.DataFrame,
        variable_name: str,
        idx_i: str,
        idx_j: str,
        grid_metadata: snail.intersection.GridDefinition,
        output_path: Path|str,
        fill_value=np.nan,
        nodata=np.nan,
    ):
    accumulated: pd.DataFrame = (
        splits.loc[:, [variable_name, idx_i, idx_j]]
        .groupby([idx_i, idx_j]).sum()
    )
    arr: np.ndarray = np.full((grid_metadata.height, grid_metadata.width), fill_value=fill_value)

    idx_js = accumulated.index.get_level_values(idx_j)
    idx_is = accumulated.index.get_level_values(idx_i)
    arr[idx_js, idx_is] = accumulated.loc[:, variable_name].values

    with rasterio.open(
        output_path,
        "w",
        driver='GTiff',
        height=arr.shape[0],
        width=arr.shape[1],
        count=1,
        dtype=arr.dtype,
        crs=grid_metadata.crs,
        transform=grid_metadata.transform,
        nodata=nodata
    ) as dst:
        dst.write(arr, 1)

In [44]:
ENSEMBLE_INFO = {
    "7": "max",
    "10": "min"
}

In [40]:
grid_metadata, _ = snail.io.read_raster_metadata(flood_tif)
for ensemble_member, desc in ENSEMBLE_INFO.items():
    member_damages = damages_by_flooded_point.loc[ensemble_member].reset_index()

    output_path = base_path / f"baseline__fluvial__ead_{desc}.tif"
    splits_to_raster(member_damages, "baseline__fluvial__ead", "flood_i", "flood_j", grid_metadata, output_path)
    print("Wrote", output_path)
    output_path = base_path / f"future__fluvial__ead_{desc}.tif"
    splits_to_raster(member_damages, "future__fluvial__ead", "flood_i", "flood_j", grid_metadata, output_path)
    print("Wrote", output_path)
    output_path = base_path / f"avoided__fluvial__ead_{desc}.tif"
    splits_to_raster(member_damages, "avoided__fluvial__ead", "flood_i", "flood_j", grid_metadata, output_path)
    print("Wrote", output_path)

Wrote ../../../processed_data/nbs-river-catchment/baseline__fluvial__ead_max.tif
Wrote ../../../processed_data/nbs-river-catchment/future__fluvial__ead_max.tif
Wrote ../../../processed_data/nbs-river-catchment/avoided__fluvial__ead_max.tif
Wrote ../../../processed_data/nbs-river-catchment/baseline__fluvial__ead_min.tif
Wrote ../../../processed_data/nbs-river-catchment/future__fluvial__ead_min.tif
Wrote ../../../processed_data/nbs-river-catchment/avoided__fluvial__ead_min.tif


### Link avoided damages to land use
for each catchment:
- find all related flooded points, total (or per sector) damage reduction
- intersect catchment geometry with landuse points
- filter to only the afforestable points
- assign damage reduction evenly to all afforestable points
  - `catchment_aff_points["damage_reduction_proportion"] = damage_reduction / len(catchment_aff_points)`

catchment_aff_points > save to parquet or append long list

In [6]:
future_damage = pd.read_parquet(base_path / "damage__future.parquet")

In [7]:
# Avoided damage
future_damage["avoided__fluvial__ead"] = (
    future_damage.baseline__fluvial__ead - future_damage.future__fluvial__ead
)
# Avoided as % of baseline damage
future_damage["avoided__fluvial__ead_pct"] = (
    future_damage["avoided__fluvial__ead"] / future_damage.baseline__fluvial__ead
) * 100

In [49]:
future_damage_lookup = future_damage.query('avoided__fluvial__ead > 1e-6').set_index(['ensemble_member', 'dem_i', 'dem_j'])
future_damage_lookup.sort_index(inplace=True)

In [9]:
pts = gpd.read_parquet(landuse_points_path)
afforestable_pts = pts.query('afforestable > 0').copy()

In [15]:
catchments_with_rp_change = gpd.read_parquet(base_path / "catchments_with_rp_change.parquet")

In [50]:
def portion_damage_reduction(subset_future_damages, catchment, afforestable_points):
    total_avoided_in_basin = subset_future_damages.avoided__fluvial__ead.sum()

    assert total_avoided_in_basin > 1e-6

    geom = catchment.geometry
    afforestable_in_catchment = points_in_polygon(geom, afforestable_points)
    total_afforestable = afforestable_in_catchment.afforestable.sum()

    assert total_afforestable > 0

    avoided_per_unit = total_avoided_in_basin / total_afforestable
    afforestable_in_catchment["avoided_ead_portion"] = afforestable_in_catchment.afforestable * avoided_per_unit

    return afforestable_in_catchment

# Sense check with example
catchment = catchments_with_rp_change.iloc[34]
dem_i, dem_j = catchment.dem_i, catchment.dem_j
subset_damages = future_damage_lookup.loc[("7", dem_i, dem_j)].reset_index()
print("Avoided", subset_damages.avoided__fluvial__ead.sum())
points_in_polygon(catchment.geometry, pts).sort_values(by="afforestable_m2")
portion_damage_reduction(subset_damages, catchment, afforestable_pts)

Avoided 181.26206310595126


,dem_i,dem_j,afforestable,afforestable_m2,existing_forest,existing_forest_m2,geometry,avoided_ead_portion
96964,155,1878,0.5,450.0,0.5,450.0,POINT (661084.566 703947.874),0.172139
95696,154,1878,0.5,450.0,0.5,450.0,POINT (661084.566 703977.874),0.172139
95695,154,1877,0.5,450.0,0.5,450.0,POINT (661054.566 703977.874),0.172139
95697,154,1879,0.5,450.0,0.5,450.0,POINT (661114.566 703977.874),0.172139
94435,153,1877,0.5,450.0,0.5,450.0,POINT (661054.566 704007.874),0.172139
...,...,...,...,...,...,...,...,...
55888,115,1891,0.5,450.0,0.5,450.0,POINT (661474.566 705147.874),0.172139
55887,115,1890,0.5,450.0,0.5,450.0,POINT (661444.566 705147.874),0.172139
55890,115,1893,0.5,450.0,0.5,450.0,POINT (661534.566 705147.874),0.172139
55889,115,1892,0.5,450.0,0.5,450.0,POINT (661504.566 705147.874),0.172139


#### Run for all catchments

In [83]:
damage_reduction: dict[str, np.ndarray] = {}

for ensemble_member, desc in ENSEMBLE_INFO.items():
    # pick ensemble damages
    future_damage_lookup_em = future_damage_lookup.loc[ensemble_member]
    # set up damage reduction array on grid
    damage_reduction[desc] = np.full((afforestable_pts.dem_i.max(), afforestable_pts.dem_j.max()), fill_value=0.0, dtype='float64')

    for ci in tqdm(range(len(catchments_with_rp_change)), total=len(catchments_with_rp_change), desc=ensemble_member):
        catchment = catchments_with_rp_change.iloc[ci]
        dem_i, dem_j = catchment.dem_i, catchment.dem_j
        try:
            subset_damages = future_damage_lookup_em.loc[(dem_i, dem_j)].reset_index()
        except KeyError:
            # print("No damages at", (dem_i, dem_j))
            continue

        pdr = portion_damage_reduction(subset_damages, catchment, afforestable_pts)
        # accumulate damage reduction to afforestable points
        damage_reduction[desc][pdr.dem_i, pdr.dem_j] += pdr.avoided_ead_portion

damage_reduction

7:   0%|          | 0/91468 [00:00<?, ?it/s]

10:   0%|          | 0/91468 [00:00<?, ?it/s]

{'max': array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]),
 'min': array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]])}

In [91]:
def write_arr(arr, grid_metadata, output_path, nodata=np.nan):
    with rasterio.open(
        output_path,
        "w",
        driver='GTiff',
        height=arr.shape[0],
        width=arr.shape[1],
        count=1,
        dtype=arr.dtype,
        crs=grid_metadata.crs,
        transform=grid_metadata.transform,
        nodata=nodata
    ) as dst:
        dst.write(arr, 1)

In [92]:
dem_metadata, _ = snail.io.read_raster_metadata(drainage_tif)
for desc in ENSEMBLE_INFO.values():
    write_arr(damage_reduction[desc], dem_metadata, base_path / f"damage_reduction_{desc}.tif")